# RAG on Amazon Bedrock, hands-on

**Day 9 · Agentic AI Bootcamp**

The whole pipeline runs against Amazon Bedrock with no extra infrastructure. Documents are embedded with Titan, searched with plain numpy cosine (same math the big stores use), and answered with Claude. You only need AWS credentials with Bedrock access.

What you will build, in order:
- embeddings + a local vector store, then naive RAG end to end
- advanced levers: query rewriting and LLM reranking
- CRAG (Corrective RAG) as a LangGraph loop that grades retrieval and corrects it
- Self-RAG as a LangGraph loop that decides whether to retrieve and self-checks its answer
- the managed path: Bedrock Knowledge Bases `retrieve` and `retrieve_and_generate`
- retrieval as a Strands tool, then wrapped as an AgentCore entrypoint
- a simple LLM-as-judge evaluation

Every API call is real. The only stub is the CRAG correction source (marked as a hook for a web search tool).

## How to run

**VS Code**: create a venv and select it, run `aws configure`, run the install cell, then top to bottom.
**Colab**: run the install cell, set `AWS_ACCESS_KEY_ID` / `AWS_SECRET_ACCESS_KEY` via `os.environ` or Colab secrets, run top to bottom.

**What changes in production**: IAM role not access keys, a real vector store (OpenSearch Serverless, pgvector) or a Bedrock Knowledge Base instead of the in-memory matrix, region and model from config, PII filtering on retrieved text, and an eval set instead of eyeballing.

In [ ]:
%pip install -q litellm langchain-litellm langgraph langchain-core numpy boto3

In [1]:
import os, re, json, litellm
import numpy as np

REGION = "us-east-1"
os.environ["AWS_REGION_NAME"] = REGION
os.environ.setdefault("AWS_DEFAULT_REGION", REGION)

GEN_MODEL = "bedrock/us.anthropic.claude-haiku-4-5-20251001-v1:0"   # generation
EMBED_MODEL = "bedrock/amazon.titan-embed-text-v2:0"                # embeddings
litellm.drop_params = True   # avoid the Bedrock temperature+top_p conflict

# Credential check (AWS creds, not an api_key)
import boto3
try:
    acct = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
    print("AWS credentials OK. Account:", acct)
except Exception as e:
    print("AWS credentials NOT found. Fix before running below. Detail:", e)

# Two helpers reused everywhere in this notebook
def chat(prompt, system=None, max_tokens=512, temperature=0.2):
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    r = litellm.completion(model=GEN_MODEL, messages=msgs,
                           max_tokens=max_tokens, temperature=temperature)
    return r.choices[0].message.content

def embed(texts):
    e = litellm.embedding(model=EMBED_MODEL, input=texts)
    return np.array([d["embedding"] for d in e.data], dtype=float)

print("Helpers ready: chat(), embed()")

AWS credentials OK. Account: 452203592848
Helpers ready: chat(), embed()


## 1. The corpus

A small set of internal docs for a fictional airline assistant, TravelMind. Note what is present (refunds, seat changes, tiers, PNR format) and what is absent (pet-in-cabin rules). That gap is used later to trigger CRAG's correction path.

In [2]:
CORPUS = [
    {"id": "refunds", "meta": {"topic": "refunds"},
     "text": "A flight cancelled by TravelMind is eligible for a full refund. The passenger has "
             "a 24 hour window to request an alternative flight before an automatic refund is issued. "
             "Refund requests are processed within 7 business days to the original payment method."},
    {"id": "seatchange", "meta": {"topic": "seats"},
     "text": "Passengers can change their seat any time after booking through Manage Booking in the app. "
             "Seat changes are free for Gold and Platinum tier members. Standard members pay a seat "
             "selection fee outside the free 24 hour post-booking window."},
    {"id": "tiers", "meta": {"topic": "loyalty"},
     "text": "TravelMind loyalty has three tiers: Silver, Gold, and Platinum. Gold members receive free "
             "seat selection, priority boarding, and waived change fees. Tier is based on annual miles flown."},
    {"id": "pnr", "meta": {"topic": "booking"},
     "text": "Every booking has a six character alphanumeric PNR, for example JX48Q2. Quote the PNR when "
             "contacting support so an agent can locate your itinerary quickly."},
    {"id": "changefee", "meta": {"topic": "fees"},
     "text": "Change fees apply to standard economy fares. Gold and Platinum members have change fees "
             "waived, though a fare difference may still apply when moving to a more expensive flight."},
    {"id": "checkin", "meta": {"topic": "checkin"},
     "text": "Online check-in opens 48 hours before departure and closes 90 minutes before departure "
             "for domestic flights and 3 hours before for international flights."},
    {"id": "baggage", "meta": {"topic": "baggage"},
     "text": "Standard fares include one carry-on bag up to 7 kg and one personal item. Checked baggage "
             "allowance depends on fare and route. Excess baggage is charged per kilogram at the airport. "
             "Fragile and oversized items follow separate handling rules and must be declared at check-in. "
             "Sports equipment such as skis and golf clubs can be booked as special baggage in advance."},
]
print(len(CORPUS), "documents loaded")

7 documents loaded


## 2. Chunking

Focused passages embed better than whole documents. Short docs stay as one chunk; the longer baggage doc splits, with overlap so a sentence on a boundary is not cut in half.

In [3]:
def chunk_text(text, size=40, overlap=8):
    words = text.split()
    if len(words) <= size:
        return [text]
    chunks, i = [], 0
    while i < len(words):
        chunks.append(" ".join(words[i:i + size]))
        i += size - overlap
    return chunks

CHUNKS = []
for doc in CORPUS:
    pieces = chunk_text(doc["text"])
    for j, piece in enumerate(pieces):
        CHUNKS.append({"id": f"{doc['id']}_{j}", "text": piece, "meta": doc["meta"]})

print(len(CHUNKS), "chunks total")
print("Example split doc 'baggage' ->", [c["id"] for c in CHUNKS if c["id"].startswith("baggage")])

9 chunks total
Example split doc 'baggage' -> ['baggage_0', 'baggage_1']


## 3. Embed and index

Embed every chunk once, stack into a matrix. Dimension is read from the response, not hardcoded.

In [4]:
MATRIX = embed([c["text"] for c in CHUNKS])
print("Index shape:", MATRIX.shape, "( chunks x embedding_dim )")
assert MATRIX.shape[0] == len(CHUNKS) and MATRIX.shape[1] > 0
print("[OK] Vector index built entirely on Bedrock Titan embeddings.")

Index shape: (9, 1024) ( chunks x embedding_dim )
[OK] Vector index built entirely on Bedrock Titan embeddings.


## 4. Retrieve

Cosine similarity between the query vector and every chunk, return the top k.

In [5]:
def retrieve(query, k=3, min_score=-1.0):
    qv = embed([query])[0]
    sims = MATRIX @ qv / (np.linalg.norm(MATRIX, axis=1) * np.linalg.norm(qv) + 1e-9)
    order = np.argsort(-sims)[:k]
    hits = []
    for idx in order:
        if sims[idx] >= min_score:
            c = CHUNKS[idx]
            hits.append({"id": c["id"], "text": c["text"], "score": float(sims[idx])})
    return hits

for h in retrieve("Can I get money back if my flight is cancelled?", k=3):
    print(f"{h['score']:.3f}  [{h['id']}]  {h['text'][:70]}...")
print("[OK] Retrieval returns the refund chunk first.")

0.531  [refunds_0]  A flight cancelled by TravelMind is eligible for a full refund. The pa...
0.290  [seatchange_0]  Passengers can change their seat any time after booking through Manage...
0.266  [changefee_0]  Change fees apply to standard economy fares. Gold and Platinum members...
[OK] Retrieval returns the refund chunk first.


## 5. Naive RAG, end to end

Retrieve, ground the prompt (answer only from context, allow "I do not know", cite ids), generate. Baseline to ship and measure before adding anything.

In [6]:
def rag_answer(query, k=3):
    hits = retrieve(query, k=k)
    ctx = "\n\n".join(f"[{h['id']}] {h['text']}" for h in hits)
    ans = chat(
        "Answer ONLY from the context below. Cite the [id] of any passage you use. "
        "If the answer is not in the context, say you do not know.\n\n"
        f"Context:\n{ctx}\n\nQuestion: {query}",
        max_tokens=300,
    )
    return ans, hits

ans, _ = rag_answer("Are seat changes free for Gold members?")
print("Q1 (in corpus):\n", ans)

ans2, _ = rag_answer("What is the pet-in-cabin weight limit?")
print("\nQ2 (not in corpus, should refuse):\n", ans2)
print("\n[OK] Grounded answer for known info, honest refusal for unknown.")

Q1 (in corpus):
 Yes, seat changes are free for Gold members. [seatchange_0] states that "Seat changes are free for Gold and Platinum tier members." This is also confirmed in [tiers_0], which notes that "Gold members receive free seat selection."

Q2 (not in corpus, should refuse):
 I do not know. The context provided does not contain information about pet-in-cabin weight limits.

[OK] Grounded answer for known info, honest refusal for unknown.


## 6. Advanced levers: query rewriting and LLM reranking

Add a lever only to fix a measured failure. Rewriting sharpens a vague question. Reranking rescues a good chunk that vector search ranked low, using the model as a second, sharper judge.

In [7]:
def rewrite_query(q):
    return chat(
        "Rewrite the question to be explicit and keyword-rich for document search. "
        f"Return only the rewritten query.\n\nQuestion: {q}",
        max_tokens=40, temperature=0.2,
    ).strip()

def rerank(query, hits):
    scored = []
    for h in hits:
        s = chat(
            "Rate from 0 to 10 how relevant this passage is to the question. Return only a number.\n"
            f"Question: {query}\nPassage: {h['text']}",
            max_tokens=5, temperature=0.0,
        )
        m = re.search(r"\d+", s)
        scored.append((int(m.group()) if m else 0, h))
    scored.sort(key=lambda x: -x[0])
    return [h for _, h in scored]

vague = "waived fees?"
print("Rewritten:", rewrite_query(vague))

wide = retrieve("fees for changing a flight", k=5)
print("\nBefore rerank:", [h["id"] for h in wide])
print("After rerank: ", [h["id"] for h in rerank("Do Gold members pay change fees?", wide)][:3])
print("[OK] Rewriting and reranking run on the same Bedrock model.")

Rewritten: What are the policies and procedures for waiving fees, fee waivers, and fee exemptions?

Before rerank: ['changefee_0', 'seatchange_0', 'refunds_0', 'checkin_0', 'tiers_0']
After rerank:  ['changefee_0', 'tiers_0', 'seatchange_0']
[OK] Rewriting and reranking run on the same Bedrock model.


## 7. CRAG (Corrective RAG) as a LangGraph loop

CRAG assumes retrieval can fail, so it grades the retrieved docs, then routes. If nothing is relevant, it corrects by rewriting the query and searching again.

```
retrieve -> grade -> (relevant?) -> generate
                          |
                          -> correct (rewrite + search) -> generate
```

The correction source here is a broadened re-retrieval over the same corpus, with a clearly marked hook where a real web search tool (Tavily, or AgentCore Web Search) would go.

In [8]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END

class CragState(TypedDict):
    question: str
    documents: List[dict]
    relevant: List[dict]
    generation: str
    route: str
    attempts: int

def crag_retrieve(state):
    return {"documents": retrieve(state["question"], k=4), "attempts": state.get("attempts", 0)}

def crag_grade(state):
    q = state["question"]
    kept = []
    for d in state["documents"]:
        v = chat(f"Question: {q}\n\nPassage: {d['text']}\n\n"
                 "Is this passage relevant and sufficient to help answer the question? "
                 "Answer one word: yes or no.", max_tokens=5, temperature=0.0).strip().lower()
        if v.startswith("y"):
            kept.append(d)
    return {"relevant": kept, "route": "generate" if kept else "correct"}

def crag_correct(state):
    # Correction. In production call a web search tool here (Tavily / AgentCore Web Search).
    # To keep the notebook runnable, broaden retrieval on a rewritten query over the same corpus.
    rq = chat("Rewrite this question to be broader and keyword-rich for search. "
              f"Return only the query.\n\nQuestion: {state['question']}",
              max_tokens=40, temperature=0.2).strip()
    broader = retrieve(rq, k=4)          # HOOK: swap for web_search(rq)
    return {"relevant": broader, "attempts": state.get("attempts", 0) + 1}

def crag_generate(state):
    ctx = "\n\n".join(f"[{d['id']}] {d['text']}" for d in state["relevant"]) or "(no passages)"
    ans = chat("Answer ONLY from the context. Cite the [id]s you use. "
               "If the context does not answer it, say you do not know.\n\n"
               f"Context:\n{ctx}\n\nQuestion: {state['question']}", max_tokens=350)
    return {"generation": ans}

g = StateGraph(CragState)
g.add_node("retrieve", crag_retrieve)
g.add_node("grade", crag_grade)
g.add_node("correct", crag_correct)
g.add_node("generate", crag_generate)
g.add_edge(START, "retrieve")
g.add_edge("retrieve", "grade")
g.add_conditional_edges("grade", lambda s: s["route"],
                        {"generate": "generate", "correct": "correct"})
g.add_edge("correct", "generate")
g.add_edge("generate", END)
crag = g.compile()

print("== In-corpus question ==")
o = crag.invoke({"question": "What is the refund window for a cancelled flight?", "attempts": 0})
print("route:", o["route"], "| relevant:", len(o["relevant"]), "\n", o["generation"])

print("\n== Weak-match question (triggers correction) ==")
o2 = crag.invoke({"question": "How many kilograms can a cabin pet weigh?", "attempts": 0})
print("route:", o2["route"], "| relevant:", len(o2["relevant"]), "\n", o2["generation"])
print("\n[OK] CRAG graded retrieval and took the corrective branch when nothing matched.")

/Users/akash-at-work/Documents/IBS Agentic AI and AWS GenAI Training/Day-10 - Strands/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


== In-corpus question ==
route: generate | relevant: 1 
 According to the context [refunds_0], the refund window for a cancelled flight is **24 hours**. During this time, the passenger can request an alternative flight before an automatic refund is issued.

== Weak-match question (triggers correction) ==
route: correct | relevant: 4 
 I do not know. The context provided does not contain information about cabin pet weight limits.

[OK] CRAG graded retrieval and took the corrective branch when nothing matched.


## 8. Self-RAG as a LangGraph loop

Self-RAG makes retrieval conditional and self-checked. It decides whether to retrieve at all, filters relevant passages, generates, then reflects on whether the answer is supported. If not, it rewrites and retries, capped to avoid looping.

```
decide -> retrieve -> grade -> generate -> reflect -> (supported?) -> END
   |                                            |
   -> direct -> END                             -> rewrite -> retrieve
```

In [9]:
class SelfRagState(TypedDict):
    question: str
    do_retrieve: str
    documents: List[dict]
    relevant: List[dict]
    generation: str
    supported: str
    attempts: int

def sr_decide(state):
    v = chat("Does answering this need company/external documents, or is general knowledge enough? "
             f"Question: {state['question']}\nAnswer one word: retrieve or direct.",
             max_tokens=5, temperature=0.0).strip().lower()
    return {"do_retrieve": "retrieve" if "retriev" in v else "direct"}

def sr_direct(state):
    return {"generation": chat(state["question"], max_tokens=200), "supported": "yes"}

def sr_retrieve(state):
    return {"documents": retrieve(state["question"], k=4)}

def sr_grade(state):
    q = state["question"]
    kept = [d for d in state["documents"]
            if chat(f"Question: {q}\nPassage: {d['text']}\nRelevant? yes or no.",
                    max_tokens=5, temperature=0.0).strip().lower().startswith("y")]
    return {"relevant": kept}

def sr_generate(state):
    ctx = "\n\n".join(f"[{d['id']}] {d['text']}" for d in state["relevant"]) or "(no relevant passages)"
    ans = chat("Answer ONLY from the context, cite [id]s, say you do not know if absent.\n\n"
               f"Context:\n{ctx}\n\nQuestion: {state['question']}", max_tokens=350)
    return {"generation": ans, "attempts": state.get("attempts", 0) + 1}

def sr_reflect(state):
    ctx = "\n\n".join(d["text"] for d in state["relevant"])
    v = chat(f"Context:\n{ctx}\n\nAnswer:\n{state['generation']}\n\n"
             "Is the answer fully supported by the context? yes or no.",
             max_tokens=5, temperature=0.0).strip().lower()
    return {"supported": "yes" if v.startswith("y") else "no"}

def sr_rewrite(state):
    rq = chat(f"Rewrite for better retrieval, return only the query: {state['question']}",
              max_tokens=40, temperature=0.2).strip()
    return {"question": rq}

def sr_route_reflect(state):
    if state["supported"].startswith("y") or state.get("attempts", 0) >= 2:
        return "end"
    return "retry"

g = StateGraph(SelfRagState)
g.add_node("decide", sr_decide)
g.add_node("direct", sr_direct)
g.add_node("retrieve", sr_retrieve)
g.add_node("grade", sr_grade)
g.add_node("generate", sr_generate)
g.add_node("reflect", sr_reflect)
g.add_node("rewrite", sr_rewrite)
g.add_edge(START, "decide")
g.add_conditional_edges("decide", lambda s: s["do_retrieve"],
                        {"retrieve": "retrieve", "direct": "direct"})
g.add_edge("direct", END)
g.add_edge("retrieve", "grade")
g.add_edge("grade", "generate")
g.add_edge("generate", "reflect")
g.add_conditional_edges("reflect", sr_route_reflect, {"retry": "rewrite", "end": END})
g.add_edge("rewrite", "retrieve")
selfrag = g.compile()

print("== Needs retrieval ==")
a = selfrag.invoke({"question": "How do I change my seat after booking?", "attempts": 0})
print("decision:", a["do_retrieve"], "| supported:", a.get("supported"), "\n", a["generation"])

print("\n== General knowledge, skips retrieval ==")
b = selfrag.invoke({"question": "What is 2 + 2?", "attempts": 0})
print("decision:", b["do_retrieve"], "\n", b["generation"])
print("\n[OK] Self-RAG skipped retrieval when unneeded and self-checked when it did retrieve.")

== Needs retrieval ==
decision: direct | supported: yes 
 # Changing Your Seat After Booking

Here are the general steps:

## Online/Self-Service
1. **Log into your account** on the airline/transport website
2. **Find your booking** using your confirmation number
3. **Select "Manage Booking"** or similar option
4. **Choose "Change Seat"** and pick a new one from the available map
5. **Confirm and save** your changes

## By Phone
- Call customer service with your confirmation number
- Ask an agent to change your seat
- They may charge a fee depending on the seat type

## At the Airport
- Visit the check-in counter or kiosk
- Ask to change your seat (subject to availability)
- Usually free, but premium seats may have fees

## Important Notes
- **Availability** varies by flight and how close to departure
- **Fees** may apply for preferred/premium

== General knowledge, skips retrieval ==
decision: direct 
 2 + 2 = 4

[OK] Self-RAG skipped retrieval when unneeded and self-checked when it d

## 9. The managed path: Bedrock Knowledge Bases

If AWS runs the index for you, retrieval and generation are two API calls. This cell only runs when you set a real `KB_ID`; the code shape is production-accurate. Note the old `citation` field is deprecated in favour of `retrievedReferences`.

In [12]:
KB_ID = "VSA3SCCQ7K"

if KB_ID:
    rt = boto3.client("bedrock-agent-runtime", region_name=REGION)
    br = boto3.client("bedrock-runtime", region_name=REGION)
    MODEL = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
    q = "What is the refund window?"

    # 1) Retrieve only  (managed KB path)
    r = rt.retrieve(
        knowledgeBaseId=KB_ID,
        retrievalQuery={"text": q},
        retrievalConfiguration={
            "managedSearchConfiguration": {
                "numberOfResults": 4,
                # "rerankingModelType": "NONE",  # skip managed reranking (faster/cheaper)
            }
        },
    )
    chunks = [it["content"]["text"] for it in r["retrievalResults"]]
    for it in r["retrievalResults"]:
        print(round(it["score"], 3), it["content"]["text"][:80])

    # 2) Generate yourself  (managed KB blocks retrieve_and_generate)
    context = "\n\n".join(chunks)
    prompt = (f"Answer only from the context. If the answer is not in it, say so.\n\n"
              f"Context:\n{context}\n\nQuestion: {q}")
    resp = br.converse(
        modelId=MODEL,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 300, "temperature": 0},
    )
    print("\nAnswer:", resp["output"]["message"]["content"][0]["text"])
else:
    print("KB_ID not set. Set it to run the managed Knowledge Base path.")

# what changes in production:
# - IAM: retrieve needs bedrock:Retrieve; converse needs bedrock:InvokeModel
#   (Converse is NOT a distinct IAM action) on the inference profile + base model
# - no hardcoded region/model; inject via env or config
# - add retries (botocore Config, adaptive) + handle ThrottlingException at load

0.446 Sheet
0.203 Sheet
Sheet
Allotment Type*	Station Code	Product	Commodity	Agent	Segment OD	Flig
0.187 Sheet
Sheet
Allotment Type*	Station Code	Product	Commodity	Agent	Segment OD	Flig

Answer: The refund window is not mentioned in the provided context. The context contains information about allotment types, station codes, products, commodities, agents, flight segments, days of week, intervals, and utilization percentages, but does not include any information about refund windows.


## 10. RAG as a Strands tool

Turning retrieval into a tool lets the agent choose to look things up only when a question needs it. This is the shape AgentCore deploys, and it is Self-RAG's "do I need to retrieve" instinct expressed as tool choice.

In [11]:
from strands import Agent, tool
from strands.models.litellm import LiteLLMModel

@tool
def retrieve_context(query: str) -> str:
    """Search the internal TravelMind knowledge base and return the most relevant passages."""
    hits = retrieve(query, k=3)
    return "\n\n".join(f"[{h['id']}] {h['text']}" for h in hits)

rag_agent = Agent(
    model=LiteLLMModel(model_id=GEN_MODEL, params={"max_tokens": 500, "temperature": 0.2}),
    tools=[retrieve_context],
    system_prompt=("Use retrieve_context for questions about TravelMind policies. "
                   "Answer from retrieved passages and cite the [id]s. Say you do not know if absent."),
)
print(rag_agent("Do Gold members pay a fee to change their flight?"))
print("\n[OK] Strands agent retrieved on demand and answered with citations.")


Tool #1: retrieve_context
No, Gold members do not pay a fee to change their flight. According to TravelMind policy, **change fees are waived for Gold members** [changefee_0]. 

However, it's important to note that while the change fee itself is waived, a **fare difference may still apply** if you're changing to a more expensive flight [changefee_0].No, Gold members do not pay a fee to change their flight. According to TravelMind policy, **change fees are waived for Gold members** [changefee_0]. 

However, it's important to note that while the change fee itself is waived, a **fare difference may still apply** if you're changing to a more expensive flight [changefee_0].


[OK] Strands agent retrieved on demand and answered with citations.


## 11. RAG on AgentCore (deploy, not run here)

The next cell writes an AgentCore entrypoint that wraps the Strands RAG agent. It is not executed in the notebook. In production the tool would call a Bedrock Knowledge Base `retrieve`, your vector store, or a Lambda exposed through AgentCore Gateway as an MCP tool.

In [ ]:
%%writefile rag_agentcore_entrypoint.py
# Deploy target: Amazon Bedrock AgentCore Runtime. NOT run inside the notebook.
# Deploy:
#   pip install bedrock-agentcore bedrock-agentcore-starter-toolkit strands-agents litellm
#   agentcore configure --entrypoint rag_agentcore_entrypoint.py
#   agentcore launch
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models.litellm import LiteLLMModel

def search(query, k=3):
    # PRODUCTION: replace with a Bedrock Knowledge Base retrieve, your vector store,
    # or a Lambda exposed via AgentCore Gateway. Returns a list of {"id","text"}.
    raise NotImplementedError("Wire this to your retrieval backend before deploy.")

@tool
def retrieve_context(query: str) -> str:
    """Search the knowledge base and return the most relevant passages."""
    hits = search(query, k=3)
    return "\n\n".join(f"[{h['id']}] {h['text']}" for h in hits)

agent = Agent(
    model=LiteLLMModel(model_id="bedrock/us.anthropic.claude-haiku-4-5-20251001-v1:0"),
    tools=[retrieve_context],
    system_prompt="Answer from retrieved passages and cite the [id]s. Say you do not know if absent.",
)

app = BedrockAgentCoreApp()

@app.entrypoint
def invoke(payload):
    """AgentCore passes the request payload; return a dict."""
    return {"result": str(agent(payload.get("prompt", "Hello")))}

if __name__ == "__main__":
    app.run()   # serves /invocations and /ping on port 8080

AgentCore RAG steps:
1. Prove the RAG agent answers locally (section 10).
2. Wrap it in `BedrockAgentCoreApp` with `@app.entrypoint` (the file above).
3. Point `search()` at a Bedrock Knowledge Base `retrieve`, your vector store, or a Gateway MCP tool.
4. `agentcore configure --entrypoint rag_agentcore_entrypoint.py`, then `agentcore launch`.
5. Invoke with `boto3.client("bedrock-agentcore").invoke_agent_runtime(...)`, session id at least 16 chars.
6. Add AgentCore Memory for history and Observability traces, without changing agent logic.

## 12. Evaluation: LLM-as-judge

RAG that is not measured is quietly failing. Two cheap signals: is the answer supported by the retrieved context (faithfulness), and does it address the question (answer relevance).

In [13]:
def judge(question, answer, context):
    faith = chat(f"Context:\n{context}\n\nAnswer:\n{answer}\n\n"
                 "Is the answer fully supported by the context? yes or no.",
                 max_tokens=5, temperature=0.0).strip().lower()
    rel = chat(f"Question: {question}\nAnswer: {answer}\n"
               "Does the answer address the question? yes or no.",
               max_tokens=5, temperature=0.0).strip().lower()
    return {"faithful": faith.startswith("y"), "relevant": rel.startswith("y")}

q = "Are seat changes free for Gold members?"
ans, hits = rag_answer(q)
ctx = "\n\n".join(h["text"] for h in hits)
print("Answer:", ans)
print("Judge:", judge(q, ans, ctx))
print("[OK] Faithfulness and relevance scored by an LLM judge. Swap in a labelled set for real eval.")

Answer: Yes, seat changes are free for Gold members. [seatchange_0] states that "Seat changes are free for Gold and Platinum tier members." This is also confirmed in [tiers_0], which notes that "Gold members receive free seat selection."
Judge: {'faithful': True, 'relevant': True}
[OK] Faithfulness and relevance scored by an LLM judge. Swap in a labelled set for real eval.


## Recap: pick the right RAG

- Naive first, always. Measure before adding anything.
- Advanced levers (rewrite, rerank, hybrid, compression) fix specific measured failures, one at a time.
- Self-RAG for mixed questions and a grounding self-check.
- CRAG when the store is patchy and freshness matters.
- Managed Knowledge Bases to ship fast; retrieval-as-a-tool to put RAG inside an agent on AgentCore.

Two things carry from today: LiteLLM is the one wire to every model. RAG is the library card that keeps the model current and honest. Both sit under the agents you built all week.